The following code will help me prepare the annotations for SAM2 fine-tuning.

First we will match the annotations to the correct tomograms and make sure they line up.

Then we will save the annotations in the correct format. (JPEG for the image and NUMPY arrays for the masks)

In [28]:
import numpy as np
import pandas as pd
import os
import cv2
import json
import shutil
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import ipywidgets as widgets
from IPython.display import HTML
import SimpleITK as sitk
from cryoet_data_portal import Client, Run, Annotation
from tqdm import tqdm
from cryoet_data_portal import Tomogram
import mrcfile
import traceback  # Add this import to handle exceptions

In [14]:
source_path = "/home/matiasgp/groups/fslg_imagseg/nobackup/archive/segmentation_data/3d_segmentations"
tomo_path = "/home/matiasgp/groups/fslg_imagseg/nobackup/archive/segmentation_data/SegData/run_6071.mha"

In [15]:
def load_mha_as_numpy(mha_path):
    """
    Load a .mha file and convert it to a NumPy array.
    
    Parameters:
    - mha_path: Path to the .mha file.
    
    Returns:
    - A NumPy array containing the image data from the .mha file.
    """
    # Read the mha file using SimpleITK
    image = sitk.ReadImage(mha_path)
    
    # Convert the image to a NumPy array
    numpy_array = sitk.GetArrayFromImage(image)
    
    return numpy_array

In [16]:
def plot_in_notebook(array, num_slices=10, save_path=None, custom_name=None, speed_factor=2, show_in_notebook=False):
    """
    Plot a 3D numpy array as animated slices with average of specified number of slices.
    The animations are 30% smaller and use grayscale, and only the shortest axis is plotted.
    
    Parameters:
    - array: 3D numpy array to plot
    - num_slices: Number of slices to average in each frame
    - save_path: Directory path to save the animations as videos. If None, videos are not saved.
    - custom_name: Custom name to be included in the saved video file name and the plot title.
    - speed_factor: A multiplier to control the speed of the animation (higher value means faster animation).
    - show_in_notebook: Boolean flag to control whether the animation is displayed in the notebook or not.
    """
    
    # Determine the shortest axis
    shortest_axis = np.argmin(array.shape)
    
    def update_plot(frame, axis, ax):
        ax.cla()  # Clear the current plot
        
        if axis == 0:
            start_slice = max(0, frame - num_slices // 2)
            end_slice = min(array.shape[axis], frame + num_slices // 2 + 1)
            slices = array[start_slice:end_slice, :, :]
            slice_ = np.mean(slices, axis=0)
            xlabel, ylabel = 'x', 'y'
        elif axis == 1:
            start_slice = max(0, frame - num_slices // 2)
            end_slice = min(array.shape[axis], frame + num_slices // 2 + 1)
            slices = array[:, start_slice:end_slice, :]
            slice_ = np.mean(slices, axis=1)
            xlabel, ylabel = 'z', 'y'
        elif axis == 2:
            start_slice = max(0, frame - num_slices // 2)
            end_slice = min(array.shape[axis], frame + num_slices // 2 + 1)
            slices = array[:, :, start_slice:end_slice]
            slice_ = np.mean(slices, axis=2)
            xlabel, ylabel = 'z', 'x'
        
        im = ax.imshow(slice_, cmap='gray', animated=True, origin='lower', vmin=0, vmax=1)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        
        # Set the plot title using custom_name if provided, else use a default title
        plot_title = f'{custom_name}' if custom_name else f'Axis {axis} ({"z" if axis == 0 else "x" if axis == 1 else "y"}) Slice {frame}'
        ax.set_title(plot_title)
        
        return [im]

    # Plot only along the shortest axis
    fig, ax = plt.subplots(figsize=(6, 4))
    
    # Calculate the interval in milliseconds, with speed_factor adjusting the speed
    interval = max(10, int(100 / speed_factor))  # 100ms is the default, and higher speed_factor means shorter interval
    
    ani = FuncAnimation(
        fig, update_plot, frames=array.shape[shortest_axis], 
        fargs=(shortest_axis, ax), blit=True, repeat=False, interval=interval
    )
    plt.close(fig)  # Close the figure to prevent it from displaying statically

    # Save the animation if save_path is provided
    if save_path is not None:
        # Ensure the save_path directory exists
        os.makedirs(save_path, exist_ok=True)
        axis_names = ['z', 'x', 'y']
        
        # Set the custom name or default to 'animation_axis'
        base_name = f'animation_axis_{axis_names[shortest_axis]}'
        if custom_name:
            base_name += f'_{custom_name}'
        
        video_filename = os.path.join(save_path, f'{base_name}.mp4')
        ani.save(video_filename, writer='ffmpeg', dpi=100)
        print(f"Saved animation for axis {axis_names[shortest_axis]} to {video_filename}")

    # Optionally display the animation in the notebook
    if show_in_notebook:
        output = widgets.Output()
        with output:
            display(HTML(ani.to_jshtml()))
        display(output)

In [17]:
def get_all_file_paths(directory):
    file_paths = []
    
    # Walk through the directory
    for root, dirs, files in os.walk(directory):
        # Add each file path to the list if it ends with .mha
        for file in files:
            if file.endswith('.mha'):
                file_path = os.path.join(root, file)
                file_paths.append(file_path)
    
    return file_paths

In [18]:

def parse_annotation_paths(file_paths):
    """
    Parse annotation file paths to extract DatasetID and RunID.
    Only numbers are extracted for both DatasetID and RunID.
    
    Args:
        file_paths: List of file paths to parse
        
    Returns:
        List of dictionaries containing filepath, DatasetID, and RunID
    """
    parsed_data = []
    
    for filepath in file_paths:
        # Get the filename from the path
        filename = os.path.basename(filepath)
        
        # Extract DatasetID (only numbers from parent directory name)
        dataset_id = ''.join(filter(str.isdigit, os.path.basename(os.path.dirname(filepath))))
        
        # Extract RunID (assuming it's the last digits before .mha)
        run_id = ''.join(filter(str.isdigit, filename.split('.')[0]))
        
        parsed_data.append({
            'annotation_path': filepath,
            'CZI Dataset ID': dataset_id,
            'CZI Run ID': run_id
        })
    
    return parsed_data

In [19]:
def save_array_as_frames(array, output_dir):
    """
    Save a 3D numpy array as sequential JPEG frames.
    
    Args:
        array: 3D numpy array with shape (frames, height, width) or (height, width, frames)
        output_dir: Directory path where frames will be saved
    """
    import os
    import cv2
    import numpy as np

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Determine which axis is shortest (assumed to be frames)
    min_axis = np.argmin(array.shape)
    
    # Rearrange array if needed so frames are first dimension
    if min_axis != 0:
        array = np.moveaxis(array, min_axis, 0)
    
    # Save each frame as a JPEG
    for i in range(array.shape[0]):
        frame = array[i]
        # Scale to 0-255 range and convert to uint8
        frame = (frame * 255).astype(np.uint8)
        # Format frame number with leading zeros
        frame_name = f"{i:05d}.jpg"
        frame_path = os.path.join(output_dir, frame_name)
        # Save frame as JPEG
        cv2.imwrite(frame_path, frame)

In [20]:
def histogram_equalization_3d(image):
    """
    Apply histogram equalization to a 3D array.
    
    Args:
    - image (np.ndarray): 3D array representing the image or volume.
    
    Returns:
    - image_equalized (np.ndarray): Histogram-equalized 3D array.
    """
    # Flatten the 3D image array and calculate histogram
    hist, bins = np.histogram(image.flatten(), bins=256, range=[0, 1])

    # Calculate cumulative distribution function (CDF)
    cdf = hist.cumsum()
    cdf_normalized = cdf / cdf.max()  # Normalize CDF

    # Use linear interpolation of the CDF to find new pixel values
    image_equalized = np.interp(image.flatten(), bins[:-1], cdf_normalized)

    # Reshape the flattened image back to the original 3D shape
    image_equalized = image_equalized.reshape(image.shape)
    
    return image_equalized

In [21]:
def min_max_normalize(array):
    # Convert to float32 to prevent overflow issues
    array = array.astype(np.float32)
    
    min_val = np.min(array)
    max_val = np.max(array)
    
    if min_val == max_val:
        # Avoid division by zero if the array contains a single unique value
        return np.zeros(array.shape, dtype=np.float32)
    
    normalized_array = (array - min_val) / (max_val - min_val)
    return normalized_array

In [43]:
def load_tomogram_file(file_path):
    """Load a tomogram from an MRC file."""
    with mrcfile.open(file_path, permissive=True, header_only=False) as mrc:
        return mrc.data

def process_tomogram(array):
    """Process tomogram array with normalization and histogram equalization."""
    array = min_max_normalize(array)
    array = histogram_equalization_3d(array)
    return array

def save_frames(array, output_dir):
    """Save 3D array as individual frame images."""
    os.makedirs(output_dir, exist_ok=True)
    for i in range(array.shape[0]):
        frame = array[i]
        frame = (frame * 255).astype(np.uint8)
        cv2.imwrite(os.path.join(output_dir, f"frame_{i:04d}.png"), frame)

def download_and_process_tomograms(annotations_data, source_file_path):
    """Download tomograms and process them into frame images."""
    client = Client()
    
    # Create directories
    tomograms_dir = os.path.join(source_file_path, "tomograms")
    frames_dir = os.path.join(source_file_path, "frames")
    os.makedirs(tomograms_dir, exist_ok=True)
    os.makedirs(frames_dir, exist_ok=True)
    
    # Process each annotation
    for entry in tqdm(annotations_data, desc="Processing tomograms"):
        try:
            dataset_id = int(entry['CZI Dataset ID'])
            run_id = entry['CZI Run ID']
            
            # Get tomogram
            runs = Run.find(client, query_filters=[Run.id == run_id, Run.dataset.id == dataset_id])
            tomo = Tomogram.find(client, query_filters=[Tomogram.name == runs[0].name])
            
            if not runs:
                print(f"No tomogram found for Run {run_id}")
                continue
                
            print(f"\nFound run: {runs[0].name}")
            
            # Setup paths
            dataset_dir = os.path.join(tomograms_dir, f"dataset_{dataset_id}")
            run_dir = os.path.join(dataset_dir, str(run_id))
            
            # Clean up any existing directory or file
            if os.path.exists(run_dir):
                if os.path.isdir(run_dir):
                    shutil.rmtree(run_dir)
                else:
                    os.remove(run_dir)
            
            # Create the run directory
            os.makedirs(run_dir, exist_ok=True)
            
            # Download tomogram
            print(f"Downloading to: {run_dir}")
            tomo[0].download_mrcfile(dest_path=run_dir)
            
            # Set up the destination path for the MRC file
            dest_path = os.path.join(run_dir, f"{tomo[0].name}.mrc")
            print(f"Processing: {dest_path}")
            # Process tomogram
            print("Processing tomogram...")
            tomo_array = load_tomogram_file(dest_path)
            tomo_array = process_tomogram(tomo_array)
            
            # Save frames
            frames_output_dir = os.path.join(frames_dir, f"dataset_{dataset_id}_{run_id}")
            save_frames(tomo_array, frames_output_dir)
            
            print(f"Successfully processed tomogram for Dataset {dataset_id}, Run {run_id}")
            
        except Exception as e:
            print(f"Error processing Dataset {dataset_id}, Run {run_id}: {str(e)}")
            print(traceback.format_exc())

In [44]:
annotations = get_all_file_paths(source_path)
annotations = parse_annotation_paths(annotations)
print(annotations)


# Example usage:
download_and_process_tomograms(annotations[0:1], source_path)


[{'annotation_path': '/home/matiasgp/groups/fslg_imagseg/nobackup/archive/segmentation_data/3d_segmentations/segmentations/dataset_10161/membrane_8379.mha', 'CZI Dataset ID': '10161', 'CZI Run ID': '8379'}, {'annotation_path': '/home/matiasgp/groups/fslg_imagseg/nobackup/archive/segmentation_data/3d_segmentations/segmentations/dataset_10084/membrane_6100.mha', 'CZI Dataset ID': '10084', 'CZI Run ID': '6100'}, {'annotation_path': '/home/matiasgp/groups/fslg_imagseg/nobackup/archive/segmentation_data/3d_segmentations/segmentations/dataset_10084/membrane_6086.mha', 'CZI Dataset ID': '10084', 'CZI Run ID': '6086'}, {'annotation_path': '/home/matiasgp/groups/fslg_imagseg/nobackup/archive/segmentation_data/3d_segmentations/segmentations/dataset_10084/membrane_6088.mha', 'CZI Dataset ID': '10084', 'CZI Run ID': '6088'}, {'annotation_path': '/home/matiasgp/groups/fslg_imagseg/nobackup/archive/segmentation_data/3d_segmentations/segmentations/dataset_10084/membrane_6097.mha', 'CZI Dataset ID': '

Processing tomograms:   0%|          | 0/1 [00:00<?, ?it/s]

15766
Run ID: 8324
Dataset ID: 10158
Run Name: ycw2014-07-18-25
Run ID: 8325
Dataset ID: 10158
Run Name: ycw2014-07-18-26
Run ID: 8326
Dataset ID: 10158
Run Name: ycw2014-07-18-27
Run ID: 10403
Dataset ID: 10234
Run Name: mba2010-03-02-8
Run ID: 10404
Dataset ID: 10234
Run Name: mba2010-03-02-9
Run ID: 14077
Dataset ID: 10301
Run Name: 15042022_BrnoKrios_Arctis_grid9_Position_32
Run ID: 14246
Dataset ID: 10302
Run Name: 02122021_BrnoKrios_Arctis_lam3_pos1
Run ID: 3917
Dataset ID: 10057
Run Name: dga2017-09-29-19
Run ID: 3920
Dataset ID: 10057
Run Name: dga2017-09-29-22
Run ID: 2426
Dataset ID: 10080
Run Name: dga2016-08-11-12
Run ID: 1792
Dataset ID: 10044
Run Name: mka2019-10-25-53
Run ID: 6003
Dataset ID: 10083
Run Name: dga2019-12-05-25
Run ID: 2302
Dataset ID: 10079
Run Name: dga2017-09-05-18
Run ID: 3923
Dataset ID: 10057
Run Name: dga2017-09-29-6
Run ID: 3924
Dataset ID: 10057
Run Name: dga2017-09-30-10
Run ID: 3925
Dataset ID: 10057
Run Name: dga2017-09-30-11
Run ID: 3926
Datase

Processing tomograms:   0%|          | 0/1 [00:17<?, ?it/s]


KeyboardInterrupt: 